In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import pickle
import math
import ast
from scipy.stats import median_abs_deviation, hypergeom, mannwhitneyu
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
# Saving plots with editable text
plt.rcParams['pdf.fonttype'] = 42  # TrueType fonts (editable text)

In [ ]:
import sys
# Ensure this analysis directory is importable regardless of kernel CWD
_here = '/projects/bhdw/asachan/methods/FIREFate/multiome_dynamic_regulation/py_scripts/analysis'
if _here not in sys.path:
    sys.path.insert(0, _here)

import dictys
from utils_custom import *
from pseudotime_curves import *
from episodic_dynamics import *
from config import *

In [ ]:
import importlib
import firefate.utils.plots, firefate.utils.custom
import temporal_clustering
# reload firefate helpers first, then temporal_clustering so it re-binds fresh names
importlib.reload(firefate.utils.plots)
importlib.reload(firefate.utils.custom)
importlib.reload(temporal_clustering)
from temporal_clustering import StateFrequency, TFForceWaves, RegulatoryPhases, TFForceValidation

In [ ]:
config = Config()

In [ ]:
# Load data
dictys_dynamic_object = dictys.net.dynamic_network.from_file('/work/nvme/bhdw/asachan/data_files/firefate/bcell/outs/dynamic.h5')

In [ ]:
PB_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(98, 147, 1)) + [2]
GC_fate_window_indices = [1] + list(range(97, 3, -1)) + [0] + list(range(147, 193, 1)) + [3]
PB_post_bifurcation_window_indices = [0] + list(range(98, 147, 1)) + [2]
GC_post_bifurcation_window_indices = [0] + list(range(147, 193, 1)) + [3]

In [ ]:
# Define distinct colors for better visibility
colors_cell_count = {
    'ActB-1': '#87CEFA',     # lightskyblue
    'ActB-2': '#1E90FF',     # dodgerblue
    'ActB-3': '#00008B',     # darkblue
    'ActB-4': '#9370DB',     # mediumorchid
    'GC-1': '#7BDE7B',       # custom light green
    'GC-2': '#008000',       # green
    'PB-2': '#BB3636',       # custom red
    'earlyActB': '#008080',   # teal
    'earlyPB': '#F08080'   # lightcoral
}

## Phasing regulatory links

In [ ]:
# PB -> 3 phases: switch boundaries are the termination pseudotimes of ActB-4 then earlyPB.
# A link lands in phase 1 if its softmax peak <= ActB-4 termination, phase 2 if between the
# two terminations, phase 3 if after earlyPB termination.
for state in ['ActB-4', 'earlyPB']:
    print(state, 'termination pseudotime:',
          sf_pb.termination_pseudotime(state, PB_post_bifurcation_window_indices, method='threshold'))

# The 66 enriched links act on both branches; xval tagged each with the lineage where its
# force was strongest. For PB phase assignment, only classify the enriched links whose
# winning branch is PB (don't assume all of ss_firefate_combined is PB).
pb_enriched_links = xval_df.loc[
    (xval_df['group'] == 'enriched') & (xval_df['branch'] == 'PB'), 'link'
].tolist()
print(f"{len(pb_enriched_links)} PB-branch enriched links used for PB phase assignment")

pb_phase_assignments = RegulatoryPhases(waves_pb).classify_phases_from_states(
    sf_pb,
    PB_post_bifurcation_window_indices,
    boundary_states=['ActB-4', 'earlyPB'],
    termination_method='threshold',
    threshold_frac=0.1,
    links=pb_enriched_links,
)
display(pb_phase_assignments)

In [ ]:
# Per-(branch, phase) comparison: enriched vs size-matched random links, one group of
# boxes per branch-qualified phase (PB 1-3, GC 1-2). Each point is one link's abs max TF
# force (max_t |force(t)|). split_by_phase reuses xval's selector / enriched links /
# random pool and only adds the phase split. Switch boundaries are the cell-state
# termination pseudotimes per lineage (PB: ActB-4, earlyPB; GC: ActB-3).
pb_switches = [sf_pb.termination_pseudotime(s, PB_post_bifurcation_window_indices,
                                            method='threshold')
               for s in ['ActB-4', 'earlyPB']]
gc_switches = [sf_pb.termination_pseudotime('ActB-3', GC_post_bifurcation_window_indices,
                                            method='threshold')]

validation = xval.split_by_phase({'PB': pb_switches, 'GC': gc_switches})
validation_df = validation.run(exclude='tf_and_target', random_state=0)
display(validation_df)

fig, ax = validation.plot(ylabel='Abs max TF force')
# fig.savefig(os.path.join(config.OUTPUT_FOLDER, 'phase_validation.pdf'), bbox_inches='tight', dpi=300)
plt.show()